In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from datetime import datetime
import os
import getpass

In [7]:

password = getpass.getpass("Enter MySQL password: ")

# Creating engine pointing to turbofan_db
engine = create_engine("mysql+pymysql://root:" + password + "@localhost:3306/turbofan_db")

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print(f"Connected to: {result.fetchone()[0]}")


Connected to: turbofan_db


In [8]:
columns = ['engine_id', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

#loading all 4 datasets

datasets = {}

for i in range(1,5):
    train = pd.read_csv(f'../data/raw/train_FD00{i}.txt', sep = '\s+', header = None, names=columns)
    test = pd.read_csv(f'../data/raw/test_FD00{i}.txt', sep = '\s+', header = None, names=columns)
    rul = pd.read_csv(f'../data/raw/RUL_FD00{i}.txt', sep = '\s+', header = None, names=['RUL'])

    train['dataset'] = f'FD00{i}'
    train['split_type'] = 'train'
    test['dataset'] = f'FD00{i}'
    test['split_type'] = 'test'

    datasets[f'FD00{i}'] = {'train': train, 'test': test, 'rul': rul}
    print(f"FD00{i} loaded - Train: {train.shape}, Test:{test.shape}")

FD001 loaded - Train: (20631, 28), Test:(13096, 28)
FD002 loaded - Train: (53759, 28), Test:(33991, 28)
FD003 loaded - Train: (24720, 28), Test:(16596, 28)
FD004 loaded - Train: (61249, 28), Test:(41214, 28)


In [9]:
#loading all datasets into MySQL engine_data table

for name, data in datasets.items():
    data['train'].to_sql('engine_data', con=engine, if_exists='append', index=False)

    data['test'].to_sql('engine_data', con=engine, if_exists='append', index=False)

    print(f'{name} pushed to MySQL')

print("\nAll datasets loaded into MYSQL")

FD001 pushed to MySQL
FD002 pushed to MySQL
FD003 pushed to MySQL
FD004 pushed to MySQL

All datasets loaded into MYSQL


In [10]:
#Calculating RUL for training data and pushing to engine_RUL table

for name,data in datasets.items():
    train = data['train'].copy()

    #Max cycle per engine
    max_cycles = train.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']

    #merging and calculating RUL
    train = train.merge(max_cycles, on='engine_id')
    train['RUL'] = train['max_cycle'] - train['cycle']

    #preparing RUL dataframe
    rul_df = train[['dataset', 'engine_id', 'max_cycle', 'cycle', 'RUL']].copy()
    rul_df.columns = ['dataset', 'engine_id', 'max_cycle', 'current_cycle', 'RUL']

    #pushing to MYSQL
    rul_df.to_sql('engine_rul', con=engine, if_exists='append', index=False)

    print(f"{name} RUL pushed to MySQL")

print("\nAll RUL values loaded")




FD001 RUL pushed to MySQL
FD002 RUL pushed to MySQL
FD003 RUL pushed to MySQL
FD004 RUL pushed to MySQL

All RUL values loaded


In [12]:
#Storing our model results in MySQL

model_results_data = {'dataset': ['FD001']*6,
                      'model_name': ['Linear Regression',
                                    'Random Forest',
                                    'XGBoost',
                                    'XGBoost Tuned',
                                    'Gradient Boosting',
                                    'Best RF Capped'
                                    ],
                    'RMSE': [43.49, 34.27, 36.38, 37.62, 37.65, 15.51],
                    'MAE': [33.36, 23.59, 25.76, 26.80, 26.92, 10.89],
                    'R2': [0.586, 0.743, 0.710, 0.690, 0.690, 0.858],
                    'run_date': [datetime.now()] * 6
}

results_df = pd.DataFrame(model_results_data)
results_df.to_sql('model_results', con=engine, if_exists='append', index=False)

print("Model results pushed to MySQL")
print(results_df)

Model results pushed to MySQL
  dataset         model_name   RMSE    MAE     R2                   run_date
0   FD001  Linear Regression  43.49  33.36  0.586 2026-03-06 20:40:20.707688
1   FD001      Random Forest  34.27  23.59  0.743 2026-03-06 20:40:20.707688
2   FD001            XGBoost  36.38  25.76  0.710 2026-03-06 20:40:20.707688
3   FD001      XGBoost Tuned  37.62  26.80  0.690 2026-03-06 20:40:20.707688
4   FD001  Gradient Boosting  37.65  26.92  0.690 2026-03-06 20:40:20.707688
5   FD001     Best RF Capped  15.51  10.89  0.858 2026-03-06 20:40:20.707688


In [13]:
# Verifying all tables have data
tables = ['engine_data', 'engine_rul', 'model_results']

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", engine)
    print(f"{table}: {df['count'][0]:,} rows")

engine_data: 265,256 rows
engine_rul: 160,359 rows
model_results: 6 rows
